In [ ]:
from pathlib import Path
from src.registry import ProgramManager, ProgramConfig

In [ ]:
# Cell 2: Initialize manager
manager = ProgramManager(cwd=Path("/Users/salahalzubi/cursor_projects/deep_agents/"))

# Check current git branch
print(f"Current branch: {manager._git_current_branch()}")
print(f"Using: {manager.cwd}") 

In [ ]:
# Cell 3: Create base program
base_config = ProgramConfig(
    name="base",
    parent=None,
    generation=0,
    system_prompt={
        "type": "preset",
        "preset": "claude_code",
        "append": "You are an expert analyst."
    },
    allowed_tools=["Read", "Write", "Bash", "Glob", "Grep"],
    output_format=None,
    metadata={}
).with_timestamp()

manager.create_program("base", base_config)
print(f"Created program/base")

In [ ]:
# Cell 4: Create iter-1 mutation (branches from base)
iter1_config = base_config.mutate(
    "iter-1",
    allowed_tools=base_config.allowed_tools + ["WebSearch"]
)
manager.create_program("iter-1", iter1_config, parent="base")
print(f"Created program/iter-1")
print(f"Current branch: {manager._git_current_branch()}")

In [ ]:
# Cell 5: Simulate Tool Generator writing a skill
skill_dir = manager.cwd / ".claude" / "skills" / "new-skill"
skill_dir.mkdir(parents=True, exist_ok=True)
(skill_dir / "SKILL.md").write_text("# New Skill\nCreated on iter-1 branch.")
print("Created new-skill on iter-1 branch")

# Commit the change
committed = manager.commit("Added new-skill")
print(f"Committed: {committed}")

In [ ]:
# Cell 6: Check lineage and programs
print(f"Lineage: {manager.get_lineage('iter-1')}")  # ['iter-1', 'base']
print(f"Programs: {manager.list_programs()}")       # ['base', 'iter-1']

In [ ]:
# Cell 7: Switch between branches and see skills change
manager.switch_to("base")
print(f"On: {manager.get_current_name()}")
print(f"new-skill exists: {(manager.cwd / '.claude/skills/new-skill').exists()}")  # False

In [ ]:
manager.switch_to("iter-1")
print(f"On: {manager.get_current_name()}")
print(f"new-skill exists: {(manager.cwd / '.claude/skills/new-skill').exists()}")  # True

In [ ]:
# Cell 8: Mark frontier
manager.mark_frontier("iter-1")
print(f"Frontier: {manager.get_frontier()}")

In [ ]:
# Cell 9: Cleanup - switch back to main
manager._git_checkout("main")
print(f"Back on: {manager._git_current_branch()}")

In [ ]:
# Uncomment to delete test branches:
manager.discard("iter-1")
manager.discard("base")